# 📖 Storybook Studio — Kaggle launcher
Runs the full stack on one Kaggle GPU VM: **Ollama** (story writer) + **Forge** (illustrator) + **FastAPI backend serving the React app**.
One Quick Tunnel on `:8000` = the public app URL. `Internet: ON`, `Accelerator: GPU T4 x2`.


In [ ]:
# 0 — CONFIG
REPO_URL = "https://github.com/koredeycode/illustrated-story-generator"
LLM_MODEL = "qwen3:8b"
WORK = "/kaggle/working/storybook"
BOOKS = "/kaggle/working/books"
print("repo:", REPO_URL)


In [ ]:
# 1 — GPU check
!nvidia-smi --query-gpu=name,memory.total,memory.used,utilization.gpu --format=csv


In [ ]:
# 2 — system deps: zstd (Ollama installer), Node 20 (frontend build), cloudflared
!apt-get update -qq && apt-get install -y -qq zstd curl
!curl -fsSL https://ollama.com/install.sh | sh
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null 2>&1 && apt-get install -y -qq nodejs
!node --version
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
!cloudflared --version


In [ ]:
# 3 — clone the app repo
import os, subprocess
if not os.path.exists(WORK):
    subprocess.run(["git", "clone", REPO_URL, WORK], check=True)
else:
    subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)
!ls $WORK


In [ ]:
# 4 — backend deps + Forge + Forge deps (py3.12-safe order)
!pip install -q -r $WORK/backend/requirements.txt
!git clone https://github.com/lllyasviel/stable-diffusion-webui-forge /kaggle/working/forge 2>/dev/null || (cd /kaggle/working/forge && git pull --ff-only)
!pip install -q "setuptools<81" wheel
import subprocess
for line in open("/kaggle/working/forge/requirements_versions.txt"):
    line = line.strip()
    if not line or line.startswith("#") or line.startswith("-") or line.startswith(("setuptools==", "scikit-image==")):
        continue
    subprocess.run(["pip", "install", "-q", line], check=True)
!pip install -q "scikit-image==0.22.0"
!pip install -q "numpy==1.26.2"
print("deps done")


In [ ]:
# 5 — start Ollama detached + pull model
import subprocess, time, requests
subprocess.Popen(["ollama", "serve"], stdout=open("/tmp/ollama.log","a"), stderr=subprocess.STDOUT, start_new_session=True)
for i in range(12):
    try:
        requests.get("http://127.0.0.1:11434/api/ps", timeout=5); print("ollama up"); break
    except Exception:
        print(f"[{i}] waiting for ollama..."); time.sleep(5)
else:
    raise SystemExit("ollama never came up — see /tmp/ollama.log")
!ollama pull $LLM_MODEL
print(requests.get("http://127.0.0.1:11434/api/ps", timeout=30).json())


In [ ]:
# 6 — build the React frontend (~1-2 min)
!(cd $WORK/frontend && npm install --no-audit --no-fund 2>&1 | tail -2)
!(cd $WORK/frontend && npm run build 2>&1 | tail -5)
!ls $WORK/frontend/dist


In [ ]:
# 7 — start backend detached on :8000
import subprocess, time, requests, os
env = dict(os.environ, DATA_DIR=BOOKS, LLM_MODEL=LLM_MODEL)
subprocess.Popen(["uvicorn", "main:app", "--host", "127.0.0.1", "--port", "8000"],
                 cwd=f"{WORK}/backend", stdout=open("/tmp/storybook.log","a"),
                 stderr=subprocess.STDOUT, start_new_session=True, env=env)
for i in range(12):
    try:
        print(requests.get("http://127.0.0.1:8000/api/health", timeout=5).json()); break
    except Exception:
        print(f"[{i}] waiting for backend..."); time.sleep(5)
else:
    raise SystemExit("backend never came up — see /tmp/storybook.log")


In [ ]:
# 8 — start Forge detached on :7860, wait until ready (first run downloads ~4GB)
import subprocess, time, requests
subprocess.Popen(["python", "launch.py", "--listen", "--port", "7860", "--skip-torch-cuda-test", "--skip-version-check", "--no-half-vae"],
                 cwd="/kaggle/working/forge", stdout=open("/tmp/forge.log","a"),
                 stderr=subprocess.STDOUT, start_new_session=True)
for i in range(40):
    try:
        r = requests.get("http://127.0.0.1:7860/sdapi/v1/sd-models", timeout=10)
        if r.status_code == 200:
            print("forge ready:", len(r.json()), "models"); break
    except Exception as e:
        print(f"[{i}] waiting...", str(e)[:80])
    time.sleep(30)
else:
    print("NOT READY — check !tail /tmp/forge.log")


In [ ]:
# 9 — public URL: Quick Tunnel to :8000. THIS URL IS THE APP 📖
import subprocess, time, re, requests
subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--http-host-header", "localhost:8000"],
                 stdout=open("/tmp/tunnel.log","a"), stderr=subprocess.STDOUT, start_new_session=True)
url = None
for _ in range(30):
    time.sleep(5)
    try:
        log = open("/tmp/tunnel.log").read()
        m = re.search(r"https://[\w-]+\.trycloudflare\.com", log)
        if m:
            url = m.group(0); break
    except FileNotFoundError:
        pass
if url is None:
    print("NO TUNNEL URL PARSED — tail of /tmp/tunnel.log:")
    try:
        print("\n".join(open("/tmp/tunnel.log").read().splitlines()[-20:]))
    except FileNotFoundError:
        print("(no tunnel log yet)")
    raise SystemExit("tunnel URL not found")
print("APP URL:", url)
import time as _t
ok = False
for i in range(6):
    _t.sleep(15)
    try:
        print(requests.get(url + "/api/health", timeout=30).json()); ok = True; break
    except Exception as e:
        print(f"[{i}] DNS propagating...", str(e)[:100])
print("LIVE ✅" if ok else "tunnel up, DNS still propagating — retry the GET above")


In [ ]:
# 10 — troubleshooting: logs + process check
!ps aux | grep -E "ollama|uvicorn|launch.py|cloudflared" | grep -v grep
!echo "--- storybook (backend) ---" && tail -20 /tmp/storybook.log
!echo "--- forge ---" && tail -10 /tmp/forge.log
!echo "--- tunnel ---" && tail -5 /tmp/tunnel.log


In [ ]:
# 11 — wiring dry-run (no GPU used): backend routes + frontend build present
import requests, os
print(requests.get("http://127.0.0.1:8000/api/health", timeout=30).json())
print(requests.get("http://127.0.0.1:8000/api/styles", timeout=30).json())
print("frontend:", os.path.exists(f"{WORK}/frontend/dist/index.html"))
print("open the APP URL from cell 9 and generate a 1-chapter test book")
